# CIMA PCA 整理版

这个版本把原来的流程收成几步：

1. VCF 转 pgen  
2. 样本名统一成 `BGE编号`  
3. 和 meta 做交集  
4. PCA 前过滤 + LD pruning  
5. 跑 PCA  
6. 把 PCA 合并回 meta，并整理一个画图表

小结果统一存成 `csv / tsv / txt`，不存 parquet。

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ===== 路径设置（相对项目根目录）=====
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and PROJECT_ROOT.name != "CIMA_multiomics_regulation":
    PROJECT_ROOT = PROJECT_ROOT.parent

if PROJECT_ROOT.name != "CIMA_multiomics_regulation":
    raise RuntimeError("未找到项目根目录 CIMA_multiomics_regulation，请在项目目录内运行该脚本。")

ROOT = PROJECT_ROOT
RAW_VCF = ROOT / "data/raw/CIMA/Gene/CIMA_phasing.merged"
META_CSV = ROOT / "data/processed/CIMA/meta_inventory/basic_id_sex_age.csv"

GENO_DIR = ROOT / "data/processed/genotype"
INTER_DIR = ROOT / "data/processed/sample_intersection"
META_DIR = ROOT / "data/processed/meta"
QC_DIR = META_DIR / "origin_qc"

GENO_DIR.mkdir(parents=True, exist_ok=True)
INTER_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

print(ROOT)
print(RAW_VCF)
print(META_CSV)

/data/work/CIMA_multiomics_regulation
/data/work/CIMA_multiomics_regulation/data/raw/CIMA/Gene/CIMA_phasing.merged
/data/work/CIMA_multiomics_regulation/data/processed/CIMA/meta_inventory/basic_id_sex_age.csv


## 1. 转 pgen，并生成重命名表

这里统一把原始样本名：

`E-B21106356138-4_E-B21106356138-4`

改成：

`E-B21106356138`

同时会检查新 ID 是否重复。

In [3]:
!bcftools query -l {RAW_VCF} > {GENO_DIR / 'genotype_samples.txt'}

!plink2 \
  --vcf {RAW_VCF} \
  --make-pgen \
  --out {GENO_DIR / 'CIMA_raw'}

psam = pd.read_csv(GENO_DIR / "CIMA_raw.psam", sep=r"\s+")

if "#IID" in psam.columns:
    iid_col = "#IID"
elif "IID" in psam.columns:
    iid_col = "IID"
else:
    raise ValueError(f"没找到 IID 列: {psam.columns.tolist()}")

if "#FID" in psam.columns:
    fid_col = "#FID"
elif "FID" in psam.columns:
    fid_col = "FID"
else:
    fid_col = None

iid = psam[iid_col].astype(str).str.strip()
fid = psam[fid_col].astype(str).str.strip() if fid_col else pd.Series(["0"] * len(psam), index=psam.index)

new_id = (
    iid.str.split("_").str[0]
       .str.replace(r"-4$", "", regex=True)
       .str.strip()
)

update = pd.DataFrame({
    "oldFID": fid,
    "oldIID": iid,
    "newFID": new_id,
    "newIID": new_id,
})

dup = update[update["newIID"].duplicated(keep=False)].copy()
if not dup.empty:
    raise ValueError("newIID 出现重复，先不要继续 update-ids。")

update.to_csv(GENO_DIR / "update_ids.tsv", sep="\t", index=False, header=False)
update[["oldIID", "newIID"]].to_csv(GENO_DIR / "update_ids_check.csv", index=False)

display(update.head())
print("n =", len(update))

bcftools: error while loading shared libraries: libcrypto.so.1.0.0: cannot open shared object file: No such file or directory
PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_raw.log.
Options in effect:
  --make-pgen
  --out /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_raw
  --vcf /data/work/CIMA_multiomics_regulation/data/raw/CIMA/Gene/CIMA_phasing.merged.vcf.gz

Start time: Wed Apr 15 14:07:43 2026
385581 MiB RAM detected, ~110854 available; reserving 110790 MiB for main
workspace.
Using up to 96 threads (change this with --threads).
--vcf: 13698278 variants scanned.
--vcf: 13696k variants converted.
/data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_raw-temporary.pgen
+
/data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_raw-temporary.pvar.zst
+
/

,oldFID,oldIID,newFID,newIID
0,0,E-B21100458292-4_E-B21100458292-4,E-B21100458292,E-B21100458292
1,0,E-B21103279967-4_E-B21103279967-4,E-B21103279967,E-B21103279967
2,0,E-B21105477143-4_E-B21105477143-4,E-B21105477143,E-B21105477143
3,0,E-B21106356138-4_E-B21106356138-4,E-B21106356138,E-B21106356138
4,0,E-B21106792844-4_E-B21106792844-4,E-B21106792844,E-B21106792844


n = 443


## 2. 更新样本名，并和 meta 求交集

这里统一按 `BGE编号` 做 join key。  
后面 PCA 合并也继续用 `BGE编号`，不要切到 `CIMA_ID`。

In [6]:
!mkdir /data/work/CIMA_multiomics_regulation/data/processed/sample_intersection
!plink2 \
  --pfile {GENO_DIR / 'CIMA_raw'} \
  --update-ids {GENO_DIR / 'update_ids.tsv'} \
  --make-pgen \
  --out {GENO_DIR / 'CIMA_BGEID'}

meta = pd.read_csv("/data/work/CIMA_multiomics_regulation/data/processed/CIMA/meta_inventory/basic_id_sex_age.csv")
psam = pd.read_csv(GENO_DIR / "CIMA_BGEID.psam", sep=r"\s+")

# 统一 meta 里的样本 ID 列名
if "BGE_ID" in meta.columns:
    meta = meta.rename(columns={"BGE_ID": "BGE编号"})
elif "BGE编号" not in meta.columns:
    raise ValueError(f"meta 缺少 BGE_ID/BGE编号 列，当前列为: {meta.columns.tolist()}")

if "#IID" in psam.columns:
    iid_col = "#IID"
elif "IID" in psam.columns:
    iid_col = "IID"
else:
    raise ValueError(f"psam 没找到 IID 列: {psam.columns.tolist()}")

meta_ids = set(meta["BGE编号"].astype(str).str.strip())
geno_ids = set(psam[iid_col].astype(str).str.strip())

common = sorted(meta_ids & geno_ids)
meta_only = sorted(meta_ids - geno_ids)
geno_only = sorted(geno_ids - meta_ids)

pd.Series(common).to_csv(INTER_DIR / "meta_genotype_common_BGEID.txt", index=False, header=False)
pd.Series(meta_only).to_csv(INTER_DIR / "meta_only_BGEID.txt", index=False, header=False)
pd.Series(geno_only).to_csv(INTER_DIR / "geno_only_BGEID.txt", index=False, header=False)

keep = pd.DataFrame({"FID": common, "IID": common})
keep.to_csv(INTER_DIR / "meta_genotype_common_BGEID.keep", sep="\t", index=False, header=False)

summary = pd.DataFrame({
    "item": ["meta_unique", "geno_unique", "common", "meta_only", "geno_only"],
    "n": [len(meta_ids), len(geno_ids), len(common), len(meta_only), len(geno_only)]
})
summary

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID.log.
Options in effect:
  --make-pgen
  --out /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID
  --pfile /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_raw
  --update-ids /data/work/CIMA_multiomics_regulation/data/processed/genotype/update_ids.tsv

Start time: Wed Apr 15 14:16:01 2026
385581 MiB RAM detected, ~119797 available; reserving 119733 MiB for main
workspace.
Using up to 96 threads (change this with --threads).
443 samples (0 females, 0 males, 443 ambiguous; 443 founders) loaded from
/data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_raw.psam.
13698278 variants loaded from
/data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_raw.pvar.
Note: No phenotype data present.

,item,n
0,meta_unique,467
1,geno_unique,443
2,common,443
3,meta_only,24
4,geno_only,0


## 3. 保留交集样本，做 PCA 前过滤和 LD pruning

In [4]:
!plink2 \
  --pfile "{GENO_DIR / 'CIMA_BGEID'}" \
  --keep "{INTER_DIR / 'meta_genotype_common_BGEID.keep'}" \
  --make-pgen \
  --out "{GENO_DIR / 'CIMA_BGEID_metaMatched'}"

!plink2 \
  --pfile "{GENO_DIR / 'CIMA_BGEID_metaMatched'}" \
  --chr 1-22 \
  --snps-only \
  --max-alleles 2 \
  --maf 0.05 \
  --geno 0.05 \
  --hwe 1e-6 \
  --indep-pairwise 50 5 0.2 \
  --out "{GENO_DIR / 'CIMA_pca_prune'}"

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID_metaMatched.log.
Options in effect:
  --keep /data/work/CIMA_multiomics_regulation/data/processed/sample_intersection/meta_genotype_common_BGEID.keep
  --make-pgen
  --out /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID_metaMatched
  --pfile /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID

Start time: Thu Apr 16 10:27:52 2026
2051949 MiB RAM detected, ~510119 available; reserving 510055 MiB for main
workspace.
Using up to 256 threads (change this with --threads).
443 samples (0 females, 0 males, 443 ambiguous; 443 founders) loaded from
/data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID.psam.
13698278 variants loaded from
/data/work/CIMA_multiomics_regulation/data/processed/geno

### --king-cutoff 排除近亲

In [5]:

!plink2 \
  --pfile {GENO_DIR / 'CIMA_BGEID_metaMatched'} \
  --chr 1-22 \
  --snps-only just-acgt \
  --max-alleles 2 \
  --maf 0.05 \
  --geno 0.05 \
  --hwe 1e-6 \
  --indep-pairwise 50 5 0.2 \
  --out {GENO_DIR / 'CIMA_king_prune'}

!plink2 \
  --pfile {GENO_DIR / 'CIMA_BGEID_metaMatched'} \
  --extract {GENO_DIR / 'CIMA_king_prune.prune.in'} \
  --make-king-table \
  --king-table-filter 0.0884 \
  --out {GENO_DIR / 'CIMA_related'}

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_king_prune.log.
Options in effect:
  --chr 1-22
  --geno 0.05
  --hwe 1e-6
  --indep-pairwise 50 5 0.2
  --maf 0.05
  --max-alleles 2
  --out /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_king_prune
  --pfile /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID_metaMatched
  --snps-only just-acgt

Start time: Thu Apr 16 10:28:45 2026
2051949 MiB RAM detected, ~483266 available; reserving 483202 MiB for main
workspace.
Using up to 256 threads (change this with --threads).
443 samples (0 females, 0 males, 443 ambiguous; 443 founders) loaded from
/data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID_metaMatched.psam.
11538206 out of 13698278 variants loaded from
/data/work/CIMA_multiomics_regulatio

## 4. 跑 PCA

算前 10 个 PC。  

In [8]:
!plink2 \
  --pfile {GENO_DIR / 'CIMA_BGEID_metaMatched'} \
  --extract {GENO_DIR / 'CIMA_pca_prune.prune.in'} \
  --pca 10 \
  --threads 6 \
  --out {GENO_DIR / 'CIMA_pca'}

pca = pd.read_csv(GENO_DIR / "CIMA_pca.eigenvec", sep=r"\s+", header=None)
if str(pca.iloc[0, 0]).strip() == "#FID":
    pca = pca.iloc[1:].copy()

pca.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, pca.shape[1] - 1)]
display(pca.head())

PLINK v2.0.0-a.6.9LM 64-bit Intel (29 Jan 2025)    cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_pca.log.
Options in effect:
  --extract /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_pca_prune.prune.in
  --out /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_pca
  --pca 10
  --pfile /data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID_metaMatched
  --threads 6

Start time: Thu Apr 16 10:39:11 2026
2051949 MiB RAM detected, ~677248 available; reserving 677184 MiB for main
workspace.
Using up to 6 compute threads.
443 samples (0 females, 0 males, 443 ambiguous; 443 founders) loaded from
/data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID_metaMatched.psam.
13698278 variants loaded from
/data/work/CIMA_multiomics_regulation/data/processed/genotype/CIMA_BGEID_metaMatched.pvar

,FID,IID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
1,E-B21100458292,E-B21100458292,0.0236492,-0.049793,-0.0435177,0.0474473,0.0214433,0.0604216,-0.104113,-0.0388956,0.00508508,-0.00970958
2,E-B21103279967,E-B21103279967,-0.0002974,0.00931738,-0.0435491,0.0505637,-0.0761656,0.0675841,0.0109987,-0.0454886,-0.0104663,-0.0315599
3,E-B21105477143,E-B21105477143,-0.0599028,-0.0192734,0.0907284,-0.0122821,0.0405487,0.0208956,0.057354,-0.0103729,0.0821926,0.039188
4,E-B21106356138,E-B21106356138,-0.0738468,0.00202776,-0.0124334,-0.0379208,-0.0149955,-0.0272918,-0.0131413,-0.0678569,0.0645262,0.022914
5,E-B21106792844,E-B21106792844,-0.0540476,0.0171126,-0.0946019,-0.0406531,0.0715604,0.0218957,0.00330172,0.0251091,-0.0337855,-0.0566521


## 5. 把 PCA 合并回 meta

这里有两个输出：

- `CIMA_meta_with_pca.csv`：主表
- `CIMA_plot_meta.csv`：更适合直接画图的小表

In [9]:
META_CSV = "/data/work/CIMA_multiomics_regulation/data/processed/CIMA/meta_inventory/basic_id_sex_age.csv"

meta = pd.read_csv(META_CSV)
pca = pd.read_csv(GENO_DIR / "CIMA_pca.eigenvec", sep=r"\s+", header=None)

# 统一列名
rename_map = {}
if "BGE_ID" in meta.columns and "BGE编号" not in meta.columns:
    rename_map["BGE_ID"] = "BGE编号"
if "CIMA_ID" in meta.columns and "CIMA ID" not in meta.columns:
    rename_map["CIMA_ID"] = "CIMA ID"
if "年龄" in meta.columns and "age" not in meta.columns:
    rename_map["年龄"] = "age"
meta = meta.rename(columns=rename_map)

# 检查必要列
required_cols = ["BGE编号", "SEX", "age"]
missing_cols = [c for c in required_cols if c not in meta.columns]
if missing_cols:
    raise ValueError(f"meta 缺少必要列: {missing_cols}；当前列为: {meta.columns.tolist()}")

# 处理 eigenvec
if str(pca.iloc[0, 0]).strip() == "#FID":
    pca = pca.iloc[1:].copy()

n_pc = pca.shape[1] - 2
pca.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, n_pc + 1)]

# 清理 ID
meta["BGE编号"] = meta["BGE编号"].astype(str).str.strip()
pca["IID"] = pca["IID"].astype(str).str.strip()

# 合并 PCA
meta_pca = meta.merge(
    pca.drop(columns=["FID"]),
    left_on="BGE编号",
    right_on="IID",
    how="inner"
).copy()

# 生成 plink 协变量文件
meta_pca["FID"] = meta_pca["IID"]
pc_cols = [f"PC{i}" for i in range(1, min(10, n_pc) + 1)]

covar = meta_pca[["FID", "IID", "age", "SEX"] + pc_cols].copy()
covar.to_csv(META_DIR / "CIMA_plink_covariates.tsv", sep="\t", index=False)

print("saved:", META_DIR / "CIMA_plink_covariates.tsv")
print("matched samples:", len(covar))
display(covar.head())

saved: /data/work/CIMA_multiomics_regulation/data/processed/meta/CIMA_plink_covariates.tsv
matched samples: 443


,FID,IID,age,SEX,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
0,E-B21106356138,E-B21106356138,23,0,-0.0738468,0.00202776,-0.0124334,-0.0379208,-0.0149955,-0.0272918,-0.0131413,-0.0678569,0.0645262,0.022914
1,E-B21133356716,E-B21133356716,59,1,-0.0492113,0.0438068,-0.0246685,0.0874497,-0.0259795,-0.0426053,0.0120793,-0.0270783,-0.0156669,0.0144696
2,E-B21138997257,E-B21138997257,68,1,0.0606662,-0.0741321,-0.060308,-0.00566975,-0.119269,0.0773261,-0.065876,-0.0389696,0.0219519,0.041801
3,E-B21155258684,E-B21155258684,35,0,-0.0661369,0.0936538,0.0209113,0.136489,-0.0215289,-0.0248352,-0.00744097,0.00539386,-0.0597433,0.00606499
4,E-B21296798284,E-B21296798284,29,0,0.00467278,0.0828693,0.00011957,-0.0468116,0.0740603,-0.0299751,-0.0071347,0.0389787,-0.00958845,-0.099521
